# 01 — Linear Regression
> **Repository:** ml-mental-models · **Folder:** 02_supervised_learning  
> **Notebook:** 01_linear_regression.ipynb  
> **Stack:** NumPy · Pandas · Matplotlib · Scikit-learn

---
## What this notebook covers
1. Intuition + math (loss function, normal equation, gradient descent)
2. From-scratch implementation — Normal Equation
3. From-scratch implementation — Gradient Descent
4. Sklearn implementation (Linear, Ridge, Lasso)
5. Full end-to-end project — California House Price Prediction
6. Interview Q&A

---
## Part 1 — Intuition and math

**The model:**
```
ŷ = w₁x₁ + w₂x₂ + ... + wₙxₙ + b   (matrix form: ŷ = Xw + b)
```
Training = finding `w` and `b` that minimise prediction error.

**The loss function — Mean Squared Error:**
```
MSE = (1/n) × Σ(yᵢ - ŷᵢ)²
```
Squaring makes errors positive and penalises large errors more than small ones.

**Two ways to find the optimal weights:**

*Normal Equation (exact, one-step):*
```
w = (XᵀX)⁻¹ Xᵀy
```

*Gradient Descent (iterative):*
```
∂MSE/∂w = (-2/n) × Σ xᵢ(yᵢ - ŷᵢ)
w = w - α × ∂MSE/∂w
```

**Key assumptions:**
- Linearity between X and y
- No multicollinearity (correlated features)
- Homoscedasticity (constant error variance)
- Features should be scaled for gradient descent to converge well

In [ ]:
# Cell 1 — Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Imports successful.')

---
## Part 2 — From scratch: Normal Equation

In [ ]:
# Cell 2 — Linear Regression via Normal Equation
class LinearRegressionNormal:
    """
    Linear Regression using the Normal Equation: w = (XtX)^-1 Xty
    Exact solution in one step. Use for small/medium feature counts.
    """
    def __init__(self):
        self.weights = None
        self.bias = None

    def fit(self, X, y):
        n = X.shape[0]
        X_b = np.column_stack([np.ones(n), X])   # add bias column
        A = X_b.T @ X_b
        b = X_b.T @ y
        params = np.linalg.solve(A, b)            # stable solve
        self.bias = params[0]
        self.weights = params[1:]
        return self

    def predict(self, X):
        return X @ self.weights + self.bias

    def mse(self, X, y):
        return np.mean((y - self.predict(X)) ** 2)

    def r2_score(self, X, y):
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        return 1 - ss_res / ss_tot


# Quick test on synthetic data
X_test = np.random.randn(100, 2)
true_w = np.array([3.0, -1.5])
y_test = X_test @ true_w + 2.0 + np.random.randn(100) * 0.3

m = LinearRegressionNormal().fit(X_test, y_test)
print(f'Learned weights : {m.weights.round(3)}')
print(f'True weights    : {true_w}')
print(f'Learned bias    : {m.bias:.3f}  (true: 2.0)')
print(f'MSE             : {m.mse(X_test, y_test):.4f}')
print(f'R2              : {m.r2_score(X_test, y_test):.4f}')

---
## Part 3 — From scratch: Mini-batch Gradient Descent

In [ ]:
# Cell 3 — Linear Regression via Gradient Descent
class LinearRegressionGD:
    """
    Linear Regression via mini-batch Gradient Descent.
    This is how every deep learning model trains, just simplified.
    """
    def __init__(self, lr=0.01, epochs=200, batch_size=32):
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.weights = None
        self.bias = None
        self.loss_hist = []

    def fit(self, X, y):
        n, p = X.shape
        self.weights = np.zeros(p)
        self.bias = 0.0

        for epoch in range(self.epochs):
            idx = np.random.permutation(n)
            X_s, y_s = X[idx], y[idx]

            for start in range(0, n, self.batch_size):
                Xb = X_s[start:start + self.batch_size]
                yb = y_s[start:start + self.batch_size]
                m = len(Xb)

                y_pred = Xb @ self.weights + self.bias
                error = y_pred - yb

                dw = (2 / m) * Xb.T @ error
                db = (2 / m) * error.sum()

                self.weights -= self.lr * dw
                self.bias -= self.lr * db

            full_pred = X @ self.weights + self.bias
            self.loss_hist.append(np.mean((y - full_pred) ** 2))

        return self

    def predict(self, X):
        return X @ self.weights + self.bias

    def r2_score(self, X, y):
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        return 1 - ss_res / ss_tot


scaler = StandardScaler()
X_sc = scaler.fit_transform(X_test)

gd_model = LinearRegressionGD(lr=0.05, epochs=200, batch_size=16).fit(X_sc, y_test)
ne_model = LinearRegressionNormal().fit(X_sc, y_test)

print(f'GD weights: {gd_model.weights.round(3)} | R2={gd_model.r2_score(X_sc, y_test):.4f}')
print(f'NE weights: {ne_model.weights.round(3)} | R2={ne_model.r2_score(X_sc, y_test):.4f}')
print(f'Loss epoch 1   : {gd_model.loss_hist[0]:.4f}')
print(f'Loss epoch 200 : {gd_model.loss_hist[-1]:.4f}')

In [ ]:
# Cell 4 — Plot the loss curve
plt.figure(figsize=(8, 4))
plt.plot(gd_model.loss_hist, color='#534AB7', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Gradient Descent convergence')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('gd_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 4 — Sklearn implementation

In [ ]:
# Cell 5 — Load real dataset and compare Linear, Ridge, Lasso
data = fetch_california_housing()
X, y = data.data, data.target
feat_names = data.feature_names
print(f'Dataset: {X.shape[0]} houses, {X.shape[1]} features')
print(f'Features: {feat_names}')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

models = {
    'Linear Regression': LinearRegression(),
    'Ridge (alpha=1)': Ridge(alpha=1.0),
    'Lasso (alpha=0.1)': Lasso(alpha=0.1),
}

print(f"\n{'Model':<22} | {'Train R2':>9} | {'Test R2':>8} | {'Test RMSE':>10} | {'Test MAE':>9}")
print('-' * 70)
for name, m in models.items():
    m.fit(X_train, y_train)
    tr_r2 = r2_score(y_train, m.predict(X_train))
    te_r2 = r2_score(y_test, m.predict(X_test))
    te_mse = mean_squared_error(y_test, m.predict(X_test))
    te_mae = mean_absolute_error(y_test, m.predict(X_test))
    print(f"{name:<22} | {tr_r2:>9.4f} | {te_r2:>8.4f} | {te_mse**0.5:>10.4f} | {te_mae:>9.4f}")

In [ ]:
# Cell 6 — Cross-validation and feature importance
m = LinearRegression()
cv_scores = cross_val_score(m, X_train, y_train, cv=5, scoring='r2')
print(f'CV R2 scores : {cv_scores.round(4)}')
print(f'CV mean      : {cv_scores.mean():.4f}')
print(f'CV std       : {cv_scores.std():.4f}')

m.fit(X_train, y_train)
coef_df = pd.DataFrame({
    'feature': feat_names,
    'weight': m.coef_,
    'abs_w': np.abs(m.coef_)
}).sort_values('abs_w', ascending=False)
print('\nFeature importance (by |weight|):')
print(coef_df.to_string(index=False))

---
## Part 5 — Full end-to-end project: California House Price Prediction

Full ML workflow: load -> EDA -> feature engineering -> split -> pipeline -> train -> evaluate -> save -> predict.

In [ ]:
# Cell 7 — Load and explore
housing = fetch_california_housing(as_frame=True)
df = housing.frame
print('Shape:', df.shape)
print('\nNull counts:')
print(df.isnull().sum())
print('\nCorrelation with target:')
print(df.corr()['MedHouseVal'].sort_values(ascending=False).round(3))

In [ ]:
# Cell 8 — Feature engineering
X = df.drop('MedHouseVal', axis=1).copy()
y = df['MedHouseVal'].values

X['RoomsPerPerson'] = X['AveRooms'] / X['AveOccup']
X['BedroomsRatio'] = X['AveBedrms'] / X['AveRooms']
X['PopPerHousehold'] = X['Population'] / X['AveOccup']
print(f'Features after engineering: {X.shape[1]}')
print(list(X.columns))

In [ ]:
# Cell 9 — Split, build pipelines, train, cross-validate
X_train, X_test, y_train, y_test = train_test_split(
    X.values, y, test_size=0.2, random_state=42
)

pipelines = {
    'Linear Regression': Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())]),
    'Ridge': Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))]),
    'Lasso': Pipeline([('scaler', StandardScaler()), ('model', Lasso(alpha=0.01))]),
    'ElasticNet': Pipeline([('scaler', StandardScaler()), ('model', ElasticNet(alpha=0.01, l1_ratio=0.5))]),
}

results = {}
print(f"{'Model':<20} | {'CV R2 mean':>11} | {'CV std':>7} | {'Test R2':>8} | {'Test RMSE':>10}")
print('-' * 68)
for name, pipe in pipelines.items():
    cv = cross_val_score(pipe, X_train, y_train, cv=5, scoring='r2')
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    test_r2 = r2_score(y_test, y_pred)
    test_mse = mean_squared_error(y_test, y_pred)
    results[name] = {'cv_mean': cv.mean(), 'cv_std': cv.std(), 'test_r2': test_r2, 'rmse': test_mse ** 0.5}
    print(f"{name:<20} | {cv.mean():>11.4f} | {cv.std():>7.4f} | {test_r2:>8.4f} | {test_mse**0.5:>10.4f}")

In [ ]:
# Cell 10 — Best model: detailed evaluation
best_name = max(results, key=lambda k: results[k]['test_r2'])
best_pipe = pipelines[best_name]
y_pred = best_pipe.predict(X_test)

print(f'Best model: {best_name}')
print(f'R2   : {r2_score(y_test, y_pred):.4f}')
print(f'RMSE : {mean_squared_error(y_test, y_pred)**0.5:.4f}  (avg error ${mean_squared_error(y_test,y_pred)**0.5*100:.0f}k)')
print(f'MAE  : {mean_absolute_error(y_test, y_pred):.4f}  (median error ${mean_absolute_error(y_test,y_pred)*100:.0f}k)')

print(f"\n{'Actual':>12} {'Predicted':>12} {'Error':>10}")
for actual, pred in zip(y_test[:8], y_pred[:8]):
    print(f'${actual*100:>9.0f}k  ${pred*100:>9.0f}k  {(pred-actual)*100:>+9.0f}k')

In [ ]:
# Cell 11 — Residual analysis + plot
residuals = y_test - y_pred
print(f'Mean residual : {residuals.mean():.4f}  (should be near 0)')
print(f'Std residual  : {residuals.std():.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(y_pred, residuals, alpha=0.15, s=10, color='#534AB7')
axes[0].axhline(0, color='#D85A30', linewidth=1.5)
axes[0].set_xlabel('Predicted value')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residual plot')
axes[0].grid(alpha=0.3)

axes[1].scatter(y_test, y_pred, alpha=0.15, s=10, color='#1D9E75')
axes[1].plot([y.min(), y.max()], [y.min(), y.max()], color='#D85A30', linewidth=1.5, linestyle='--')
axes[1].set_xlabel('Actual')
axes[1].set_ylabel('Predicted')
axes[1].set_title('Predicted vs Actual')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('residual_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 12 — Save and reload model, predict on new data
joblib.dump(best_pipe, 'house_price_model.pkl')
print('Model saved to house_price_model.pkl')

loaded_model = joblib.load('house_price_model.pkl')
new_house = np.array([[8.0, 25, 6.0, 1.0, 800, 2.5, 37.0, -122.0,
                        2.4, 0.167, 320.0]])
predicted_price = loaded_model.predict(new_house)[0]
print(f'Predicted price for new house: ${predicted_price*100:.0f}k')

---
## Part 6 — Interview Q&A

**Q1: What is Linear Regression?**
> A supervised algorithm modelling y as a linear function of X: ŷ = Xw + b, trained by minimising MSE.

**Q2: Why MSE as the loss function?**
> Differentiable everywhere, penalises large errors more, has one global minimum (convex).

**Q3: Assumptions of Linear Regression?**
> Linearity, no multicollinearity, homoscedasticity, normally distributed errors, independent samples.

**Q4: What is R2?**
> Proportion of variance in y explained by the model. 1=perfect, 0=no better than mean, negative=worse than mean.

**Q5: Why scale features?**
> Gradient descent converges much faster and more stably when features share similar scales.

**Q6: What is multicollinearity and the fix?**
> Highly correlated features make weights unstable. Fix: drop one feature, or use Ridge regression.

**Q7: Normal Equation vs Gradient Descent?**
> Normal Equation: exact, one-step, O(p^3), good for small feature counts. Gradient Descent: iterative, scales to large datasets.

**Q8: Train R2=0.91, Test R2=0.60 — diagnosis?**
> Overfitting (high variance). Fix: Ridge/Lasso regularisation, more data, fewer features, cross-validation.

In [ ]:
# Cell 13 — Self test
print('SELF-TEST — answer without looking up:')
qs = [
    'What does the Normal Equation compute?',
    'Why is MSE used instead of raw error?',
    'What does R2 = 0 mean?',
    'Name 3 assumptions of Linear Regression.',
    'Train R2 high, Test R2 low — what is wrong and how to fix?',
]
for i, q in enumerate(qs, 1):
    print(f'  Q{i}: {q}')